## Take Home Exercise #2 Xuanlin Liu (xl3572)

In [0]:
from pyspark.sql.functions import *
# withColumn,col, floor, datediff, current_date, avg


#### 1. [10 pts] What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.

The pit stops dataset contains one row per pit stop event, with a milliseconds column recording the duration of each stop. A single driver may have multiple pit stops in a single race. 

Group by raceId and driverId and compute the average pit stop duration per driver per race.
Then group by raceId only to find the fastest (minimum) and slowest (maximum) single pit stop across all drivers in that race.
Join these two results together so each row contains: race, driver, that driver's average stop, and the race-level fastest/slowest.

In [0]:
df_pitstop = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header = True)
display(df_pitstop)

In [0]:
df_pitstop = df_pitstop.withColumn("milliseconds", col("milliseconds").cast("long"))

# Average pit stop time per driver per race
df_avg_pitstop = (df_pitstop.groupBy("raceId", "driverId").agg(avg("milliseconds")))
# display(df_avg_pitstop)

# Fastest and slowest 
df_race_extremes = (df_pitstop.groupBy("raceId").agg(min("milliseconds"),max("milliseconds")))

# Join avg per driver with race-level extremes
df_q1 = df_avg_pitstop.join(df_race_extremes, on="raceId")

display(df_q1)

- withColumn("milliseconds", col("milliseconds").cast("long")) — the CSV reader loads all columns as strings by default. Casting to long enables numeric aggregations.
- .groupBy("raceId", "driverId").agg(avg(...)) — groups the data by race and driver, then computes the average pit stop duration for each group. The result has one row per (race, driver) pair.
- .groupBy("raceId").agg(min(...), max(...)) — collapses to one row per race and returns the minimum (fastest) and maximum (slowest) single stop time observed in that race.
- .join(df_race_extremes, on="raceId") — combines both aggregations on the shared raceId key, so each driver-level row is enriched with the race-wide fastest and slowest values.

#### 2. [20 pts] Rank order by finishing position the average time spent at the pit stop in each race.


The results.csv file contains raceId, driverId, and positionOrder.

I will reuse df_avg_pitstop from Question 1, and join result on raceId and driverId. And order by positionOrder to display each race's drivers ranked from 1st to last, alongside their average pit stop time.

In [0]:
df_result = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header = True)
display(df_result)

In [0]:
df_results_slim = (df_result.select("raceId", "driverId", "positionOrder").withColumn("positionOrder", col("positionOrder").cast("int")).withColumn("raceId", col("raceId").cast("int")))
# display(df_results_slim)

# Join average pit stop times with finishing positions, then sort
df_q2 = (df_avg_pitstop.join(df_results_slim, on=["raceId", "driverId"]).orderBy(col("raceId").cast("int"), col("positionOrder").cast("int"))
)

display(df_q2)

- .select("raceId", "driverId", "positionOrder") — narrows the results DataFrame to only the three columns needed, keeping things lean.
- .withColumn("positionOrder", col("positionOrder").cast("int")) — converts positionOrder from string to integer. Without this position "10" would appear before "2".
- .withColumn("raceId", col("raceId").cast("int")) — same reasoning for raceId, ensuring races are ordered numerically from smallest to largest.
- .join(df_results_slim, on=["raceId", "driverId"]) — links each driver's average pit stop time (from Q1) to their finishing position in that race, matching on both raceId and driverId together to avoid a cross-join.
- .orderBy(col("raceId").cast("int"), col("positionOrder").cast("int")) — sorts the final output first by race (ascending), then within each race by finishing position (ascending, 1st to last). The .cast("int") inside orderBy is a safety measure in case the cast in the earlier step did not propagate through the join.

#### 3. [20 pts] Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.


The code column has missing \\N, indicating null values. The code seems to always be the capital 3 letters of driverRef, 
I will keep where code is not null, and generate first 3 characters of driverRef and conver to capital letters. 

In [0]:
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header = True)
display(df_drivers)

In [0]:
df_drivers_code = (df_drivers.withColumn("code", when((col("code") == "\\N"), upper(substring(col("driverRef"), 1, 3))).otherwise(col("code"))))

display(df_drivers_code)

- when((col("code") == "\\N"), ...) — the condition checks for `\\N` used by the database as a null placeholder.
— substring(col, start, length) - extracts characters 1 through 3 from driverRef. 
- upper() - converts to uppercase, matching the standard 3-letter driver code format.
- .otherwise(col("code")) — if the code already exists, it is left unchanged.
- withColumn("code", ...) — overwrites the existing code column in-place with the corrected values.

#### 4. [20 pts] Who is the youngest and the oldest driver in each race? Create a new column called “Age”. Explain your definition of "age".


I define age as the driver's age on the day of the race. The number of complete years between their date of birth and the race date. 

I will Load races.csv to get each race's date. Join driver data and result with race data on raceId. Then compute the age and use window functions to find the minimum and maximum age per race.

In [0]:
df_races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header = True)
display(df_races)

In [0]:
# join drivers dataset set and laptimes dataset
df_driver_result = df_drivers.select('driverId', 'driverRef', 'forename', 'surname', 'dob', 'nationality').join(df_result, on=['driverId'])

# Keep race date and cast to date type
df_races_slim = (df_races.select("raceId", "date").withColumn("race_date", to_date(col("date"), "yyyy-MM-dd")))
# display(df_races_slim)

# Bring race date into the lap_drivers dataframe (deduplicate to one row per driver per race)
df_driver_race = (df_driver_result.select("raceId", "driverId", "driverRef", "forename", "surname", "dob").dropDuplicates(["raceId", "driverId"]).join(df_races_slim, on="raceId"))
display(df_driver_race)


In [0]:
# Compute age in years on race day
df_driver_race = df_driver_race.withColumn("Age",floor(datediff(col("race_date"), to_date(col("dob"), "yyyy-MM-dd")) / 365).cast("int"))
display(df_driver_race)

In [0]:
from pyspark.sql import Window
# Window: per race, find min and max age
race_win = Window.partitionBy("raceId")

df_q4 = (df_driver_race.withColumn("min_age_in_race", min("Age").over(race_win)).withColumn("max_age_in_race", max("Age").over(race_win)))

# Filter to only the youngest and oldest in each race
df_youngest = (df_q4.filter(col("Age") == col("min_age_in_race")).withColumn("role", lit("Youngest")))

df_oldest = (df_q4.filter(col("Age") == col("max_age_in_race")).withColumn("role", lit("Oldest")))

# union the oldest and youngest drivers
df_youngest_oldest = df_youngest.union(df_oldest).orderBy("raceId", "role")

display(df_youngest_oldest.select("raceId", "race_date", "driverId", "forename", "surname", "dob", "Age", "role"))

- to_date(col("date"), "yyyy-MM-dd") — parses the race date string into DateType so date arithmetic is possible.
- dropDuplicates(["raceId", "driverId"]) — the lap-level data has many rows per driver per race; we only need one row per driver per race to compute their age.
- datediff(race_date, dob) — returns the number of calendar days between the two dates.
- floor(... / 365) — divides by 365 to account for years, then floor() truncates to the last completed birthday, matching the intuitive definition of age.
- min("Age").over(race_win) and max("Age").over(race_win) — window functions that compute the youngest and oldest age within each race partition without collapsing rows, so we can filter directly.
- filter(col("Age") == col("min_age_in_race")) — keeps only the youngest driver per race.
- union() — stacks the youngest and oldest DataFrames vertically.

#### 5. [20 pts] At any given race, how many podiums does each driver have? create three new columns to show - on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver


"At any given race" means how many podium finishes has the driver accumulated historically up to that point. Therefore this will be a running a cumulative count, not an all-time total.

I will use the position order column from result data to map 1st, 2nd, and 3rd place. Then create a binary column for win, 2rd and 3rd place. And join with drivers for race date and order by date. Apply accumulated sum partition by driverId and order by date. 


In [0]:
# Reuse df_result from Q2 — positionOrder already represents finishing position
df_results_q5 = (df_result.select("raceId", "driverId", "positionOrder").withColumn("positionOrder", col("positionOrder").cast("int")))

# Binary indicators: positionOrder 1 = win, 2 = 2nd, 3 = 3rd
df_results_q5 = (df_results_q5.withColumn("is_win", when(col("positionOrder") == 1, 1).otherwise(0))
                 .withColumn("is_2nd", when(col("positionOrder") == 2, 1).otherwise(0))
                 .withColumn("is_3rd", when(col("positionOrder") == 3, 1).otherwise(0)))

# display(df_results_q5)

# Join with df_driver_race from Q4 to get race_date for ordering
df_results_q5 = df_results_q5.join(df_driver_race.select("raceId", "driverId", "race_date").dropDuplicates(["raceId", "driverId"]), on=["raceId", "driverId"])
# display(df_results_q5)


In [0]:

# Cumulative window per driver ordered chronologically
driver_cum_window = (Window.partitionBy("driverId").orderBy("race_date"))

df_q5 = (df_results_q5.withColumn("cumulative_wins", sum("is_win").over(driver_cum_window))
         .withColumn("cumulative_2nd",  sum("is_2nd").over(driver_cum_window))
         .withColumn("cumulative_3rd",  sum("is_3rd").over(driver_cum_window)))

# display(df_q5)
display(df_q5.select("raceId", "race_date", "driverId", "positionOrder","cumulative_wins", "cumulative_2nd", "cumulative_3rd")
        .orderBy("race_date","driverId"))

- df_result.select() — reuses the results DataFrame already loaded in Q2V.
- .withColumn("positionOrder", col("positionOrder").cast("int")) — converts positionOrder from string to integer 
- when(col("positionOrder") == 1, 1).otherwise(0) — creates a binary equal to 1 when the driver finished 1st and 0 otherwise.
- df_driver_race.select(...).dropDuplicates(["raceId", "driverId"]) — reuses df_driver_race from Q4, which already has race_date. 
- dropDuplicates ensures one row per driver per race before the join.
- Window.partitionBy("driverId").orderBy("race_date") — defines a window that resets per driver and grows from their very first race up to the current row.
- sum("is_win").over(driver_cum_window) — applies the cumulative sum within each driver's ordered window.

#### 6. [10 pts] Continue exploring the data by answering your own question.

My Question: Do older or younger drivers tend to finish higher?

F1 drivers compete across a wide age range. I want to explore whether age at race time is associated with better or worse finishing positions.

I will reuse the df_driver_rce, whichi contains the age columns and join with results. When group drivers in different age group (under 23, 23-27, 28-32, 33+). Then compute the average finishing position for each age group (lower means better result).

In [0]:
# Reuse df_driver_race (Age from Q4) and df_results_q5 (positionOrder from Q5)
df_age_position = (df_driver_race.select("raceId", "driverId", "Age").join(df_results_q5.select("raceId", "driverId", "positionOrder"), on=["raceId", "driverId"]).filter(col("positionOrder") > 0))

# Assign age group
df_age_group = df_age_position.withColumn("age_group", when(col("Age") < 23,  "Under 23")
                                          .when(col("Age") < 28, "23-27")
                                          .when(col("Age") < 33, "28-32")
                                          .otherwise("33+"))

# Average finishing position and count per age group
df_q6 = (df_age_group.groupBy("age_group").agg(avg("positionOrder").alias("avg_finishing_position"), count("*").alias("race_entries")).orderBy("avg_finishing_position"))

display(df_q6)